In [ ]:
from netCDF4 import Dataset, MFDataset
import numpy as np
from ozone_extremes_leap import *
import matplotlib.pyplot as plt
import xarray as xr 
from mpl_toolkits.basemap import Basemap
from matplotlib.pyplot import cm
import seaborn as sns
from matplotlib import rc
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def calc_parc_o3(var):
    
        m_air = 28.964/(6.022E23)
        g = 980.6
    
        if 'plev' in var.dims: 
            plev=var.plev
            level='plev'
        if 'lev' in var.dims: 
            plev=var.lev
            level='lev'
        if 'level' in var.dims: 
            plev=var.level
            level='level'
        
        
        time=var.time
        delta_p = np.zeros((len(time), len(plev)))
        
        if plev[0]<plev[len(plev)-1] and plev[len(plev)-1] <= 1000: 
            factor=100
            factor_2=1 # conversion Pa->hPa
        if plev[0]>plev[len(plev)-1] and plev[0] <=1000: 
            factor=100
            factor_2=1
        if plev[0]<plev[len(plev)-1] and plev[len(plev)-1] >1000: 
            factor=1
            factor_2=100
        if plev[0]>plev[len(plev)-1] and plev[0] >1000: 
            factor=1
            factor_2=100
        
        if plev[0]<plev[len(plev)-1]: # for pressure levels in hPa
            
            
            for levelx in range(1,len(plev)): delta_p[:,levelx].fill(plev[levelx] - plev[levelx-1])

            weights_p = xr.DataArray(delta_p*factor, dims=['time',level], coords=[time,plev]) # difference between pressure levels in Pa
 
            O3 = var * weights_p * 10/ (g * m_air)
        
            if level=='plev': O3=O3.sel(plev=slice(30*factor_2,70*factor_2)) 
            if level=='lev': O3=O3.sel(lev=slice(30*factor_2,70*factor_2))
            if level=='level': O3=O3.sel(level=slice(30*factor_2,70*factor_2))

            O3 = O3.sum(dim=level)
            O3 = O3/2.687E16  #calculate DU
            
        if plev[0]>plev[len(plev)-1]: # for pressure levels in hPa
        
            
            for levelx in range(0,len(plev)-1): delta_p[:,levelx].fill(plev[levelx] - plev[levelx+1])

            weights_p = xr.DataArray(delta_p*factor, dims=['time',level], coords=[time,plev]) # difference between pressure levels in Pa

            O3 = var * weights_p * 10/ (g * m_air)
            
            if level =='plev': O3=O3.sel(plev=slice(70*factor_2,30*factor_2)) 
            if level=='lev': O3=O3.sel(lev=slice(70*factor_2,30*factor_2)) 
            if level=='level': O3=O3.sel(level=slice(70*factor_2,30*factor_2)) 
                
            O3 = O3.sum(dim=level)
            O3 = O3/2.687E16  #calculate DU
    
        return O3.where(O3 != 0)

In [ ]:
def prepare_o3(nc_fid):
    o3=nc_fid['O3']
    o3=o3.mean(dim='lon')

    o3=calc_parc_o3(o3)

    o3=o3.sel(lat=slice(60,90))

    weights = np.cos(np.deg2rad(o3.lat))
    o3 = o3.weighted(weights)     
    o3=o3.mean(dim='lat')
    
    return o3

In [ ]:
def prepare_u(nc_fid):
    u=nc_fid['U']
    u=u.sel(plev=5000)

    u=u.sel(time=nc_fid.time.dt.month.isin([1,2,3,4,5,6]))
    u=u.mean(dim='lon')

    u=u.sel(lat=60, method='nearest')

    return u

In [ ]:
def prepare_t(nc_fid):
    t=nc_fid['T']
    t=t.mean(dim='lon')

    t=t.sel(plev=5000)

    t=t.sel(lat=slice(60,90)).min(dim='lat')

    return t

Import timeslice 2000 dataset:

In [ ]:
#WACCM INT_O3 timeslice y2000

nc_fid=xr.open_dataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc')

In [ ]:
o3=prepare_o3(nc_fid)
t=prepare_t(nc_fid)
u=prepare_u(nc_fid)

Calculate ozone minimum years and plot ozone partial column in those years:

In [ ]:
#select years with the strongest ozone minima

var=o3.sel(time=nc_fid.time.dt.month.isin([1,2,3,4,5,6]))
var_clim=var.groupby('time.dayofyear').mean()

var_o3=[]

for year in [3,8,13,14,19]:
    var_o3.append(var.sel(time=var.time.dt.year.isin([year])))

    
t=t.sel(time=t.time.dt.month.isin([1,2,3,4,5,6]))
t_clim=t.groupby('time.dayofyear').mean()

var_t=[]

for year in [3,8,13,14,19]:
    var_t.append(t.sel(time=t.time.dt.year.isin([year])))

u=u.sel(time=u.time.dt.month.isin([1,2,3,4,5,6]))
u_clim=u.groupby('time.dayofyear').mean()

var_u=[]

for year in [3,8,13,14,19]:
    var_u.append(u.sel(time=u.time.dt.year.isin([year])))

## Ozone plots for different initialisations

In [ ]:
nc_fid_0003_F=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0003/Feb/BWCN.*.nc' ,concat_dim='member', combine='nested')
nc_fid_0008_F=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb/BWCN.*.nc' ,concat_dim='member', combine='nested')
nc_fid_0013_F=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0013/Feb/BWCN.*.nc' ,concat_dim='member', combine='nested')
nc_fid_0014_F=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0014/Feb/BWCN.*.nc' ,concat_dim='member', combine='nested')
nc_fid_0019_F=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0019/Feb/BWCN.*.nc' ,concat_dim='member', combine='nested')

nc_fid_o3=[]
nc_fid_o3.append(nc_fid_0003_F)
nc_fid_o3.append(nc_fid_0008_F)
nc_fid_o3.append(nc_fid_0013_F)
nc_fid_o3.append(nc_fid_0014_F)
nc_fid_o3.append(nc_fid_0019_F)

nc_fid_0003_M=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0003/Mar/BWCN.*.nc' ,concat_dim='member', combine='nested')
nc_fid_0008_M=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar/BWCN.*.nc' ,concat_dim='member', combine='nested')
nc_fid_0013_M=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0013/Mar/BWCN.*.nc' ,concat_dim='member', combine='nested')
nc_fid_0014_M=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0014/Mar/BWCN.*.nc' ,concat_dim='member', combine='nested')
nc_fid_0019_M=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0019/Mar/BWCN.*.nc' ,concat_dim='member', combine='nested')

nc_fid_o3_M=[]
nc_fid_o3_M.append(nc_fid_0003_M)
nc_fid_o3_M.append(nc_fid_0008_M)
nc_fid_o3_M.append(nc_fid_0013_M)
nc_fid_o3_M.append(nc_fid_0014_M)
nc_fid_o3_M.append(nc_fid_0019_M)

years=['0003','0008','0013','0014','0019']

In [ ]:
o3_prep=[] #February initialisations
o3_prep_M=[] #March initialisations

for ini in range(len(nc_fid_o3)):
    o3_prep.append(prepare_o3(nc_fid_o3[ini]))
    o3_prep_M.append(prepare_o3(nc_fid_o3_M[ini]))

In [ ]:
rc('font',**{'family':'sans-serif','sans-serif':['Helvetica']})
rc('text', usetex=True)

fig, ax = plt.subplots(len(o3_prep), 1 ,figsize=(7,18), constrained_layout=True, edgecolor='k', sharex='col', sharey='row')
sns.set_style("ticks")


for i in range(len(o3_prep)):
    
    ax[i].plot(range(len(var_o3[i])), var_o3[i], color='grey', linewidth=3)
    ax[i].plot(range(len(var_clim)), var_clim , color='grey', linestyle=':', linewidth=3)


    ax[i].plot(range(31,len(o3_prep[i].time)+31), o3_prep[i].mean(dim='member'), color='royalblue',  linewidth=3, label='Feb, small pertlim')

    for mem in range(len(o3_prep[i].member)):
        ax[i].plot(range(31,len(o3_prep[i].time)+31), o3_prep[i].sel(member=mem), color='royalblue',  linewidth=0.8, alpha=0.5)
    
    ax[i].plot(range(59,len(o3_prep_M[i].time)+59), o3_prep_M[i].mean(dim='member'), color='hotpink',  linewidth=3, label='Mar, small pertlim')

    for mem in range(len(o3_prep_M[i].member)):
        ax[i].plot(range(59,len(o3_prep_M[i].time)+59), o3_prep_M[i].sel(member=mem), color='hotpink',  linewidth=0.8, alpha=0.5)
    
    
    ax[i].set_title('Year ' + str(years[i]), fontsize=22)

ax[0].set_xlim(0,150)
ax[0].legend(fontsize=22)

for j in range(len(o3_prep)):
    ax[j].set_xticks([0,31,59,89,120])
    ax[j].set_xticklabels( ['Jan','Feb','Mar','Apr','May', 'June'], fontsize=22)
    ax[j].set_ylim(70,140)
    ax[j].set_yticklabels(ax[j].get_yticks(), size = 22)

    
ax[2].set_ylabel('Partial ozone column \n 30-70 hPa (DU)', fontsize=22)

plt.savefig('/home/weiji/test code/plots/O3_evolution_min_restart.pdf')
plt.savefig('/home/weiji/test code/plots/O3_evolution_min_restart.png')

In [ ]:
## plot temperature

In [ ]:
t_prep=[]
t_prep_M=[]

for ini in range(len(nc_fid_o3)):
    t_prep.append(prepare_t(nc_fid_o3[ini]))
    t_prep_M.append(prepare_t(nc_fid_o3_M[ini]))

In [ ]:
rc('font',**{'family':'sans-serif','sans-serif':['Helvetica']})
rc('text', usetex=True)

fig, ax = plt.subplots(len(t_prep), 1 ,figsize=(7,18), constrained_layout=True, edgecolor='k', sharex='col', sharey='row')
sns.set_style("ticks")


for i in range(len(t_prep)):
    
    ax[i].plot(range(len(var_t[i])), var_t[i], color='grey', linewidth=3)
    ax[i].plot(range(len(t_clim)), t_clim , color='grey', linestyle=':', linewidth=3)


    ax[i].plot(range(31,len(t_prep[i].time)+31), t_prep[i].mean(dim='member'), color='royalblue',  linewidth=3, label='Feb, small pertlim')

    for mem in range(len(t_prep[i].member)):
        ax[i].plot(range(31,len(t_prep[i].time)+31), t_prep[i].sel(member=mem), color='royalblue',  linewidth=0.8, alpha=0.5)
    
    
    ax[i].plot(range(59,len(t_prep_M[i].time)+59), t_prep_M[i].mean(dim='member'), color='hotpink',  linewidth=3, label='Feb, small pertlim')

    for mem in range(len(t_prep_M[i].member)):
        ax[i].plot(range(59,len(t_prep_M[i].time)+59), t_prep_M[i].sel(member=mem), color='hotpink',  linewidth=0.8, alpha=0.5)
    
    
    ax[i].set_title('Year ' + str(years[i]), fontsize=22)

ax[0].set_xlim(0,150)
ax[0].legend(fontsize=22)

for j in range(len(o3_prep)):
    ax[j].set_xticks([0,31,59,89,120])
    ax[j].set_xticklabels( ['Jan','Feb','Mar','Apr','May', 'June'], fontsize=22)
    ax[j].set_ylim(180,230)
    ax[j].set_yticklabels(ax[j].get_yticks(), size = 22)

    
    
ax[2].set_ylabel('Polar cap temperature \n 50 hPa (K)', fontsize=22)

plt.savefig('/home/weiji/test code/plots/t_evolution_min_restart.pdf')
plt.savefig('/home/weiji/test code/plots/t_evolution_min_restart.png')